# Acrobot-v1

## Environment Overview

The Acrobot is a classic underactuated control problem. It consists of a two-link pendulum with only the second joint actuated. The goal is to swing the end-effector (tip of the second link) to a height at least the length of one link above the base.

Think of it as a double pendulum where you can only control the joint between the two links, but not the joint connecting to the base.

## Observation Space

The observation is a 6-dimensional continuous array representing:

1. **cos(θ₁)**: Cosine of the first joint angle
2. **sin(θ₁)**: Sine of the first joint angle  
3. **cos(θ₂)**: Cosine of the second joint angle
4. **sin(θ₂)**: Sine of the second joint angle
5. **θ₁̇**: Angular velocity of the first joint
6. **θ₂̇**: Angular velocity of the second joint

Using cos/sin representation avoids angle wrapping issues.

## Action Space

The action space is discrete with 3 possible actions:
- 0: Apply -1 torque to the second joint
- 1: Apply 0 torque to the second joint (no action)
- 2: Apply +1 torque to the second joint

## Reward Structure

- Reward of -1 for each time step until the goal is reached
- Episode terminates when the tip reaches the target height or after 500 steps
- Goal: Minimize the number of steps to reach the target

## Success Condition

The episode is successful when the tip of the second link reaches a height equal to or greater than one link length above the base.


In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import math

## Utility Functions

Helper functions to understand and work with the Acrobot state space.

In [2]:
def get_tip_position(observation):
    """
    Calculate the (x, y) position of the tip from observation
    Assuming each link has length 1
    """
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, _, _ = observation
    
    # First link endpoint
    x1 = sin_theta1
    y1 = -cos_theta1
    
    # Second link endpoint (tip)
    x2 = x1 + sin_theta2
    y2 = y1 - cos_theta2
    
    return x2, y2

def get_tip_height(observation):
    """
    Get the height of the tip above the base
    """
    _, y_tip = get_tip_position(observation)
    return y_tip + 2  # Base is at y = -2 when both links point down

def is_goal_reached(observation):
    """
    Check if the goal height (1.0) is reached
    """
    return get_tip_height(observation) >= 1.0

def get_angles_from_observation(observation):
    """
    Convert cos/sin representation back to angles (for debugging)
    """
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, _, _ = observation
    
    theta1 = math.atan2(sin_theta1, cos_theta1)
    theta2 = math.atan2(sin_theta2, cos_theta2)
    
    return theta1, theta2

def print_state_info(observation):
    """
    Print detailed state information
    """
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, vel1, vel2 = observation
    theta1, theta2 = get_angles_from_observation(observation)
    x_tip, y_tip = get_tip_position(observation)
    height = get_tip_height(observation)
    
    print(f"Angles: θ1={math.degrees(theta1):6.1f}°, θ2={math.degrees(theta2):6.1f}°")
    print(f"Velocities: θ1̇={vel1:6.2f}, θ2̇={vel2:6.2f}")
    print(f"Tip position: ({x_tip:6.3f}, {y_tip:6.3f})")
    print(f"Tip height: {height:.3f} (goal: ≥1.0)")
    print(f"Goal reached: {is_goal_reached(observation)}")

## Simple Policies

Let's start with some basic policies to understand the problem.

In [3]:
def random_policy(observation):
    """
    Random action selection
    """
    return np.random.randint(0, 3)

def energy_based_policy(observation):
    """
    Policy based on increasing system energy
    Apply torque in the direction that increases energy
    """
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, vel1, vel2 = observation
    
    # Calculate approximate energy (kinetic + potential)
    # This is a simplified version - real energy calculation is more complex
    kinetic_energy = 0.5 * (vel1**2 + vel2**2)
    potential_energy = get_tip_height(observation)
    
    # Simple heuristic: pump energy when moving, coast when high
    if potential_energy > 0.8:
        return 1  # No action when high
    elif vel2 > 0.1:
        return 2  # Pump in positive direction
    elif vel2 < -0.1:
        return 0  # Pump in negative direction
    else:
        return 1  # No action when slow

def swing_up_policy(observation):
    """
    Swing-up policy based on angular velocities and positions
    """
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, vel1, vel2 = observation
    
    # Get current tip height
    height = get_tip_height(observation)
    
    # If close to goal, try to maintain
    if height > 0.9:
        # Balance near the top
        if vel2 > 0.5:
            return 0  # Counter the velocity
        elif vel2 < -0.5:
            return 2  # Counter the velocity
        else:
            return 1  # Stay put
    
    # Swing up phase - pump energy
    if vel2 * sin_theta2 > 0:
        # Velocity and position in same direction
        return 2 if sin_theta2 > 0 else 0
    else:
        # Apply opposite torque to build momentum
        return 0 if sin_theta2 > 0 else 2

def test_policy(policy, num_episodes=10, render_mode=None, max_steps=500, verbose=False):
    """
    Test a policy on Acrobot environment
    """
    env = gym.make('Acrobot-v1', render_mode=render_mode)
    
    total_rewards = []
    episode_lengths = []
    successes = 0
    
    for episode in range(num_episodes):
        observation, info = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            action = policy(observation)
            observation, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                if is_goal_reached(observation):
                    successes += 1
                break
        
        total_rewards.append(total_reward)
        episode_lengths.append(step + 1)
        
        if verbose or num_episodes <= 5:
            height = get_tip_height(observation)
            success = is_goal_reached(observation)
            print(f"Episode {episode + 1}: Reward = {total_reward}, Steps = {step + 1}, Success = {success}, Final height = {height:.3f}")
    
    env.close()
    
    avg_reward = np.mean(total_rewards)
    avg_steps = np.mean(episode_lengths)
    success_rate = successes / num_episodes
    
    print(f"\nResults over {num_episodes} episodes:")
    print(f"Average reward: {avg_reward:.2f}")
    print(f"Average steps: {avg_steps:.2f}")
    print(f"Success rate: {success_rate:.2%}")
    
    return avg_reward, avg_steps, success_rate

In [4]:
# Test simple policies
policies = {
    'Random': random_policy,
    'Energy-based': energy_based_policy,
    'Swing-up': swing_up_policy
}

print("Testing simple policies (5 episodes each):\n")

for name, policy in policies.items():
    print(f"Testing {name} Policy:")
    test_policy(policy, num_episodes=5)
    print()

Testing simple policies (5 episodes each):

Testing Random Policy:
Episode 1: Reward = -500.0, Steps = 500, Success = False, Final height = 0.185
Episode 2: Reward = -500.0, Steps = 500, Success = False, Final height = 0.135
Episode 3: Reward = -500.0, Steps = 500, Success = False, Final height = 0.108
Episode 4: Reward = -500.0, Steps = 500, Success = False, Final height = 0.629
Episode 5: Reward = -500.0, Steps = 500, Success = False, Final height = 0.291

Results over 5 episodes:
Average reward: -500.00
Average steps: 500.00
Success rate: 0.00%

Testing Energy-based Policy:
Episode 1: Reward = -137.0, Steps = 138, Success = True, Final height = 1.536
Episode 2: Reward = -184.0, Steps = 185, Success = True, Final height = 1.885
Episode 3: Reward = -500.0, Steps = 500, Success = True, Final height = 1.170
Episode 4: Reward = -176.0, Steps = 177, Success = True, Final height = 2.201
Episode 5: Reward = -132.0, Steps = 133, Success = True, Final height = 2.699

Results over 5 episodes:


## Deep Q-Network (DQN) Implementation

Given the 6-dimensional continuous observation space, we'll use a neural network to approximate the Q-function.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class DQN(nn.Module):
    """
    Deep Q-Network for Acrobot
    """
    def __init__(self, input_size=6, hidden_size=128, output_size=3):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

class DQNAgent:
    """
    DQN Agent for Acrobot
    """
    def __init__(self, state_size=6, action_size=3, learning_rate=0.001, 
                 memory_size=10000, batch_size=64, epsilon=1.0, 
                 epsilon_decay=0.995, min_epsilon=0.01, gamma=0.99):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=memory_size)
        self.batch_size = batch_size
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        self.gamma = gamma
        
        # Neural networks
        self.q_network = DQN(state_size, 128, action_size).to(device)
        self.target_network = DQN(state_size, 128, action_size).to(device)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        
        # Copy weights to target network
        self.target_network.load_state_dict(self.q_network.state_dict())
        
    def remember(self, state, action, reward, next_state, done):
        """
        Store experience in replay buffer
        """
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state, training=True):
        """
        Choose action using epsilon-greedy policy
        """
        if training and np.random.random() < self.epsilon:
            return np.random.randint(0, self.action_size)
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_values = self.q_network(state_tensor)
        return q_values.argmax().item()
    
    def replay(self):
        """
        Train the network on a batch of experiences
        """
        if len(self.memory) < self.batch_size:
            return 0
        
        # Sample batch from memory
        batch = random.sample(self.memory, self.batch_size)
        states = torch.FloatTensor([e[0] for e in batch]).to(device)
        actions = torch.LongTensor([e[1] for e in batch]).to(device)
        rewards = torch.FloatTensor([e[2] for e in batch]).to(device)
        next_states = torch.FloatTensor([e[3] for e in batch]).to(device)
        dones = torch.BoolTensor([e[4] for e in batch]).to(device)
        
        # Current Q values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1))
        
        # Next Q values from target network
        next_q_values = self.target_network(next_states).max(1)[0].detach()
        target_q_values = rewards + (self.gamma * next_q_values * ~dones)
        
        # Compute loss
        loss = F.mse_loss(current_q_values.squeeze(), target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def update_target_network(self):
        """
        Copy weights from main network to target network
        """
        self.target_network.load_state_dict(self.q_network.state_dict())
    
    def decay_epsilon(self):
        """
        Decay exploration rate
        """
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

def train_dqn(episodes=2000, max_steps=500, update_target_freq=100):
    """
    Train DQN agent on Acrobot
    """
    env = gym.make('Acrobot-v1')
    agent = DQNAgent()
    
    rewards_per_episode = []
    steps_per_episode = []
    successes = []
    losses = []
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        episode_loss = []
        
        for step in range(max_steps):
            action = agent.act(state, training=True)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.remember(state, action, reward, next_state, done)
            
            # Train the agent
            loss = agent.replay()
            if loss > 0:
                episode_loss.append(loss)
            
            state = next_state
            total_reward += reward
            
            if done:
                success = is_goal_reached(state)
                successes.append(success)
                break
        else:
            successes.append(False)
        
        # Update target network
        if episode % update_target_freq == 0:
            agent.update_target_network()
        
        agent.decay_epsilon()
        rewards_per_episode.append(total_reward)
        steps_per_episode.append(step + 1)
        losses.append(np.mean(episode_loss) if episode_loss else 0)
        
        # Print progress
        if (episode + 1) % 200 == 0:
            recent_success_rate = np.mean(successes[-200:]) if len(successes) >= 200 else np.mean(successes)
            avg_steps = np.mean(steps_per_episode[-200:]) if len(steps_per_episode) >= 200 else np.mean(steps_per_episode)
            avg_loss = np.mean(losses[-200:]) if len(losses) >= 200 else np.mean(losses)
            print(f"Episode {episode + 1}: Success rate = {recent_success_rate:.2%}, "
                  f"Avg steps = {avg_steps:.1f}, Epsilon = {agent.epsilon:.3f}, Loss = {avg_loss:.4f}")
    
    env.close()
    return agent, rewards_per_episode, steps_per_episode, successes, losses

Using device: cuda


In [6]:
# Train the DQN agent (this will take a while)
print("Training DQN agent...")
print("This may take 10-20 minutes depending on your hardware.")
dqn_agent, dqn_rewards, dqn_steps, dqn_successes, dqn_losses = train_dqn(episodes=2000)

Training DQN agent...
This may take 10-20 minutes depending on your hardware.


/tmp/ipykernel_22129/1683281418.py:80: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  states = torch.FloatTensor([e[0] for e in batch]).to(device)


KeyboardInterrupt: 

In [ ]:
# Test the trained DQN agent
def test_dqn_agent(agent, num_episodes=10):
    """
    Test the trained DQN agent
    """
    def dqn_policy(observation):
        return agent.act(observation, training=False)
    
    return test_policy(dqn_policy, num_episodes=num_episodes, verbose=True)

print("Testing trained DQN agent:")
test_dqn_agent(dqn_agent, num_episodes=10)

## Training Progress Visualization

In [ ]:
def plot_dqn_training(rewards, steps, successes, losses, window=100):
    """
    Plot DQN training progress
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    def moving_average(data, window):
        return np.convolve(data, np.ones(window)/window, mode='valid')
    
    episodes = range(len(rewards))
    
    # Rewards
    ax1.plot(episodes, rewards, alpha=0.3, color='blue', label='Episode Reward')
    if len(rewards) >= window:
        smooth_rewards = moving_average(rewards, window)
        ax1.plot(range(window-1, len(rewards)), smooth_rewards, color='red', label=f'{window}-Episode Average')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Rewards')
    ax1.legend()
    ax1.grid(True)
    
    # Steps per episode
    ax2.plot(episodes, steps, alpha=0.3, color='green', label='Episode Steps')
    if len(steps) >= window:
        smooth_steps = moving_average(steps, window)
        ax2.plot(range(window-1, len(steps)), smooth_steps, color='orange', label=f'{window}-Episode Average')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Steps to Completion')
    ax2.set_title('Training Steps')
    ax2.legend()
    ax2.grid(True)
    
    # Success rate
    success_rate = []
    for i in range(len(successes)):
        if i < window:
            success_rate.append(np.mean(successes[:i+1]))
        else:
            success_rate.append(np.mean(successes[i-window+1:i+1]))
    
    ax3.plot(episodes, success_rate, color='purple', label=f'Success Rate ({window}-episode window)')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Success Rate')
    ax3.set_title('Training Success Rate')
    ax3.legend()
    ax3.grid(True)
    ax3.set_ylim(-0.05, 1.05)
    
    # Training loss
    if len(losses) >= window:
        smooth_losses = moving_average(losses, window)
        ax4.plot(range(window-1, len(losses)), smooth_losses, color='red', label=f'{window}-Episode Average Loss')
    ax4.plot(episodes, losses, alpha=0.3, color='brown', label='Episode Loss')
    ax4.set_xlabel('Episode')
    ax4.set_ylabel('Loss')
    ax4.set_title('Training Loss')
    ax4.legend()
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot training progress
plot_dqn_training(dqn_rewards, dqn_steps, dqn_successes, dqn_losses)

## Final Policy Comparison

In [ ]:
# Compare all policies
def dqn_policy_wrapper(observation):
    return dqn_agent.act(observation, training=False)

all_policies = {
    'Random': random_policy,
    'Energy-based': energy_based_policy, 
    'Swing-up': swing_up_policy,
    'DQN': dqn_policy_wrapper
}

print("Final Policy Comparison (50 episodes each):\n")
results = {}

for name, policy in all_policies.items():
    print(f"Testing {name} Policy:")
    avg_reward, avg_steps, success_rate = test_policy(policy, num_episodes=50, render_mode=None)
    results[name] = {'avg_reward': avg_reward, 'avg_steps': avg_steps, 'success_rate': success_rate}
    print()

# Summary comparison
print("\n" + "="*80)
print("FINAL POLICY COMPARISON")
print("="*80)
print(f"{'Policy':<15} {'Avg Reward':<12} {'Avg Steps':<12} {'Success Rate':<15}")
print("-"*80)

for name, result in results.items():
    print(f"{name:<15} {result['avg_reward']:<12.2f} {result['avg_steps']:<12.1f} {result['success_rate']:<15.1%}")

## Visualization Demo

Uncomment and run to see the trained agent in action.

In [ ]:
def visualize_acrobot(agent, episodes=2):
    """
    Visualize the trained agent solving Acrobot
    """
    env = gym.make('Acrobot-v1', render_mode='human')
    
    for episode in range(episodes):
        observation, _ = env.reset()
        total_reward = 0
        print(f"\nEpisode {episode + 1}:")
        print_state_info(observation)
        
        for step in range(500):
            action = agent.act(observation, training=False)
            observation, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            # Print periodic updates
            if step % 50 == 0:
                height = get_tip_height(observation)
                action_names = ['Left torque', 'No torque', 'Right torque']
                print(f"  Step {step:3d}: Height={height:6.3f}, Action={action_names[action]}")
            
            if terminated or truncated:
                success = is_goal_reached(observation)
                print(f"\n  Finished in {step + 1} steps. Success: {success}")
                print(f"  Total reward: {total_reward}")
                print("  Final state:")
                print_state_info(observation)
                break
            
            time.sleep(0.02)  # Small delay for visualization
    
    env.close()

# Uncomment to visualize (requires display)
# visualize_acrobot(dqn_agent, episodes=2)

## Key Insights

1. **Underactuated Control**: Acrobot demonstrates the challenges of controlling underactuated systems
2. **Energy Management**: Success requires careful energy pumping and conservation strategies
3. **Deep Reinforcement Learning**: DQN can learn complex control policies for continuous state spaces
4. **Exploration vs Exploitation**: The epsilon-greedy strategy helps balance learning and performance

## Further Improvements

- **Double DQN**: Reduce overestimation bias in Q-learning
- **Prioritized Experience Replay**: Focus learning on important transitions
- **Dueling DQN**: Separate value and advantage estimation
- **Policy Gradient Methods**: Learn policies directly (REINFORCE, Actor-Critic)
- **Continuous Control**: Adapt for continuous action spaces (DDPG, SAC)
